# 06 · Guardrails and approval

Notebook 03 ended with the agent refunding $250 for the late espresso machine with nobody's approval.
Policy P5 says any refund over $100 needs a supervisor, the model had read it, and it refunded anyway. A
rule in a prompt is advice; a **guardrail** is a check in code that the model cannot skip.

**You will learn**

- the guardrails ShipFast already has, and the ones CrewAI adds
- how a CrewAI Flow pauses for a person with `@human_feedback`
- how a paused flow is saved and later resumed with a decision
- why, in CrewAI, the refund has to become its own flow step

In [1]:
import json

from crew_agents.approval import ShipFastApprovalFlow, resume  # import crew_agents first: telemetry off
from crewai.flow.async_feedback import HumanFeedbackPending

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast

verify_model()  # stops with the command that fixes it if Ollama or the shared model is missing

# The customer message for the over-$100 runs below: order A1003, the $250 espresso machine.
LATE = "Order A1003 is more than a week late. Please cancel it and refund me."

Every run below ends by looking at the **refund ledger**, the shared record of refunds that really
happened. This small helper reads it and returns the (order, amount) pairs, or `"no refund"` when the
ledger is empty.

In [2]:
# shipfast.refunds() lists the refunds the shared tool actually issued; an empty list becomes a readable string.
def ledger():
    return [(r["order_id"], r["amount"]) for r in shipfast.refunds()] or "no refund"

## Guardrails so far

| Guardrail | Where it lives | What it stops |
|---|---|---|
| `issue_refund` refuses more than the order total, or than what is left after earlier refunds | the shared tool | a made-up amount (02), a double refund (04) |
| `max_iter` | the agent | a run that never stops (03) |
| the ledger check | ShipFast's test runner | a reply or result that claims a refund that never happened |
| a `Task` guardrail | CrewAI: `Task(guardrail=...)` | a task output that fails your own check; CrewAI sends it back for another attempt |
| **approval for refunds over $100** | **a Flow step: this notebook** | money moving without a person's yes |

The task guardrail is CrewAI's built-in output check: a function that receives the task's output and
returns `(True, output)` or `(False, "what to fix")`, and CrewAI retries the task with the feedback. It
checks what the agent *says*. Approval is different: the run must **stop and wait** for a person, possibly
for hours, before any money moves.

## Where CrewAI can pause

LangGraph can pause in the middle of a tool call. CrewAI pauses **between steps of a Flow**: a method
decorated with `@human_feedback` runs, then the flow asks a person for feedback before going on. So the
refund cannot stay a tool call buried inside the agent's loop. ShipFast's CrewAI gate splits it:

1. The agent gets a **gated** `issue_refund`. Up to $100 it runs at once. Above that it does not run: the
   request is recorded in the flow's state as pending, and the agent is told a supervisor must approve.
2. A router sends a flow with a pending refund to a `@human_feedback` step, which pauses the flow.
3. On **approve**, code issues exactly the pending refund. On **reject**, the agent is told and must
   escalate to a human.

Here is all of it, from `crew_agents/approval.py`:

```python
"""ShipFast's approval gate (SPEC.md section 3, policy P5) as a CrewAI Flow with `@human_feedback`.

CrewAI pauses between flow steps, not inside a tool call, so the gate has two parts:

1. The specialist agent gets a gated `issue_refund`. Refunds up to $100.00 run at once; a larger one is
   not run, only recorded as pending in the flow's state, and the agent is told it awaits a supervisor.
2. A router sends a flow with a pending refund to a `@human_feedback` step, which pauses the flow
   (`kickoff` returns `HumanFeedbackPending`) and saves its state with `@persist`.
3. `resume(flow_id, "approve")` restores the flow from that saved state and issues exactly the pending
   refund, in code; `resume(flow_id, "reject")` tells the agent, which must escalate to a human. CrewAI
   only resumes from saved state, so persistence is required, here a SQLite file inside the tutorial.

The feedback is mapped to "approve" or "reject" by the local model: CrewAI's default for that mapping is
an OpenAI model.
"""

from __future__ import annotations

import json
from pathlib import Path

from crewai import Agent
from crewai.flow import human_feedback
from crewai.flow.async_feedback import HumanFeedbackPending
from crewai.flow.flow import Flow, listen, router, start
from crewai.flow.persistence import SQLiteFlowPersistence, persist
from crewai.tools import tool
from pydantic import BaseModel

from agent_contracts.shipfast import tools as shipfast
from agent_contracts.shipfast.tools import APPROVAL_THRESHOLD  # policy P5, shared by all builds
from crew_agents.llm import make_llm
from crew_agents.prompts import BACKSTORY, GOAL, ROLE

FEEDBACK_LLM = make_llm()  # maps the supervisor's free-text reply onto "approve" or "reject"
FLOW_DB = Path(__file__).resolve().parents[1] / ".crewai_flows" / "flows.db"  # inside the tutorial folder
FLOW_DB.parent.mkdir(exist_ok=True)
PERSISTENCE = SQLiteFlowPersistence(str(FLOW_DB))  # where paused flows are saved, so they can be resumed


class SupervisorQueue:
    """A feedback provider that never blocks: it pauses the flow and lets the caller resume it later."""

    # CrewAI's default provider waits for an answer typed in the console; a notebook or a server has none.
    # Raising HumanFeedbackPending tells CrewAI to save the flow and return instead of waiting.
    def request_feedback(self, context, flow):
        raise HumanFeedbackPending(context=context, callback_info={"queue": "supervisors"})


# The flow's state: one customer turn, saved by @persist after each step.
class RefundCase(BaseModel):
    message: str = ""  # the customer's message for this turn
    history: list[dict] = []  # the conversation so far, as chat messages
    pending_refund: dict | None = None  # a refund over $100 waiting for a supervisor, or None
    reply: str = ""  # what the customer is told


@persist(PERSISTENCE)  # save the state after every step, so a paused flow can be restored by its id
class ShipFastApprovalFlow(Flow[RefundCase]):
    def __init__(self, llm=None, **kwargs):
        kwargs.setdefault("suppress_flow_events", True)  # no console panels for every step
        super().__init__(**kwargs)
        self.llm = llm or make_llm()

    # The specialist's tools: the shared ones, except that issue_refund is swapped for a gated version.
    def _tools(self):
        def issue_refund(order_id: str, amount: float, reason: str) -> str:
            """Issue a refund for an order. The amount cannot exceed the order total minus anything already refunded."""
            try:
                value = float(amount)
            except (TypeError, ValueError):
                value = None  # not a number: let the shared tool reject it below
            if value is not None and value > APPROVAL_THRESHOLD:
                # Over $100: do not refund. Record it in the flow's state and tell the model it is waiting.
                self.state.pending_refund = {"order_id": order_id, "amount": value, "reason": reason}
                return json.dumps({"status": "pending_approval", "order_id": order_id, "amount": value,
                                   "next_step": "Do not call issue_refund again. Tell the customer a supervisor "
                                                "must approve this refund and they will hear back shortly."})
            return shipfast.issue_refund(order_id, amount, reason)  # $100 or less: the real shared tool

        others = [f for f in shipfast.TOOLS if f.__name__ != "issue_refund"]
        return [tool(f.__name__)(f) for f in others] + [tool("issue_refund")(issue_refund)]

    # One agent turn: a new agent is given the whole history, and its reply is added to it.
    async def _agent_turn(self, text: str) -> str:
        agent = Agent(role=ROLE, goal=GOAL, backstory=BACKSTORY, tools=self._tools(), llm=self.llm, verbose=False)
        self.state.history.append({"role": "user", "content": text})
        reply = (await agent.kickoff_async(self.state.history)).raw
        self.state.history.append({"role": "assistant", "content": reply})
        return reply

    # Step 1, the entry point: kickoff runs this first. The agent answers the customer.
    @start()
    async def handle_message(self):
        self.state.reply = await self._agent_turn(self.state.message)

    # Step 2, a router: runs after handle_message, and the label it returns picks the next step.
    @router(handle_message)
    def check_refund(self):
        return "needs_approval" if self.state.pending_refund else "done"  # nothing listens for "done"

    # Step 3, on "needs_approval": @human_feedback pauses the flow for a person once this method returns.
    # The person's reply is mapped by FEEDBACK_LLM onto one of the emit labels, "approve" or "reject".
    @listen("needs_approval")
    @human_feedback(message="Approve this refund? Reply approve or reject.", emit=["approve", "reject"],
                    llm=FEEDBACK_LLM, provider=SupervisorQueue())
    def ask_supervisor(self):
        return self.state.pending_refund  # what the supervisor is shown

    # Step 4a, on "approve": code, not the model, issues exactly the refund the supervisor saw.
    @listen("approve")
    def issue_approved_refund(self):
        p = self.state.pending_refund
        result = json.loads(shipfast.issue_refund(p["order_id"], p["amount"], p["reason"]))
        self.state.pending_refund = None
        if result.get("status") == "issued":
            self.state.reply = (f"Good news: your refund of ${result['amount']:.2f} for order {result['order_id']} "
                                "has been approved and issued.")
        else:
            self.state.reply = "The approved refund could not be issued; a specialist will follow up."

    # Step 4b, on "reject": no refund. The agent is told, and must escalate to a human (policy P5).
    @listen("reject")
    async def escalate_rejected_refund(self):
        p = self.state.pending_refund
        self.state.pending_refund = None
        self.state.reply = await self._agent_turn(
            f"(Supervisor) The refund of ${p['amount']:.2f} for order {p['order_id']} was rejected. Do not "
            "refund. Call escalate_to_human, then tell the customer a specialist will follow up.")


async def resume(flow_id: str, decision: str) -> ShipFastApprovalFlow:
    """Restore a paused flow from its saved state and continue it with the supervisor's decision."""
    flow = ShipFastApprovalFlow.from_pending(flow_id, persistence=PERSISTENCE)  # load the saved state
    await flow.resume_async(decision)  # the decision is the supervisor's reply; the flow continues from step 3
    return flow
```

Three details in that code. `SupervisorQueue` is a feedback **provider** that never blocks: it raises
`HumanFeedbackPending`, which CrewAI turns into a pause instead of a console prompt (a notebook has no
console). `@persist` saves the flow's state to SQLite, because CrewAI resumes only from saved state. And
`llm=FEEDBACK_LLM`: CrewAI maps the person's free-text reply onto `approve` or `reject` with a model, and
its default for that is an OpenAI model, so the local one is set.

## Under $100: no pause

The lost running shoes (A1002) are a $45 refund. The gate lets it straight through.

In [4]:
shipfast.reset()  # start from an empty ledger and ticket list
flow = ShipFastApprovalFlow()
# kickoff_async runs the flow until it finishes or pauses; inputs are the flow's starting state.
out = await flow.kickoff_async(inputs={"message": "My order A1002 never arrived and tracking says it's lost. I want a refund."})

print("paused for approval:", isinstance(out, HumanFeedbackPending))  # False: $45 is under the limit
print("refunds issued:     ", ledger())
print("flow's reply:       ", flow.state.reply.strip()[:160])

paused for approval: False
refunds issued:      [('A1002', 45.0)]
flow's reply:        Your refund for order A1002 has been issued successfully. The total amount of $45.00 will be refunded to your original payment method. Thank you for your patien


## Over $100: the flow stops

The espresso machine (A1003) is $250. A small model does not always reach the refund on the first turn: sometimes it answers in text
and ends its turn, and then the gate has nothing to pause. So if the first turn does not attempt
the refund, the customer says "go ahead", as a real customer would (all three tutorials do this).

If even the follow-up does not reach the refund, which happens more often with the smaller models,
the helper says so and the cells below skip their step instead of failing.

When the flow pauses, `kickoff` returns a `HumanFeedbackPending` instead of a result.
Nothing has been refunded; the request waits in the flow's saved state, under the flow's id.

In [5]:
FOLLOW_UP = "Yes, please go ahead and issue the refund now."
NO_PAUSE = ("no refund attempt after the follow-up either, so the gate had nothing to pause. Smaller models do "
            "this more often; the cells below skip their step.")
SKIPPED = "skipped: the run never paused for approval (see the cell above)"


async def run_until_refund_attempt():
    """Run the late-order message in a new flow. If the model ends its turn without trying the refund (so the
    gate has nothing to pause), send a follow-up like a real customer would. Returns the flow and the pending
    approval, or None in its place if the model never tried the refund."""
    flow = ShipFastApprovalFlow()
    out = await flow.kickoff_async(inputs={"message": LATE})
    if not isinstance(out, HumanFeedbackPending):
        print("first turn ended without a refund attempt:", flow.state.reply.strip()[:140])
        history = list(flow.state.history)  # the conversation so far, carried into the next run
        flow = ShipFastApprovalFlow()
        out = await flow.kickoff_async(inputs={"message": FOLLOW_UP, "history": history})
    if not isinstance(out, HumanFeedbackPending):
        print(NO_PAUSE)
        return flow, None
    return flow, out

Now run it until the flow pauses, and look at what is waiting for the supervisor.

In [6]:
shipfast.reset()
flow, paused = await run_until_refund_attempt()  # paused is a HumanFeedbackPending (None if it never paused)

if paused is None:
    print(SKIPPED)
else:
    print("paused for approval:", isinstance(paused, HumanFeedbackPending))
    print("flow id:            ", paused.context.flow_id)  # the key a supervisor's answer is sent back under

    # The refund the gate recorded instead of issuing: this is what a supervisor would be asked to approve.
    pending = flow.state.pending_refund
    print("pending refund:")
    print("  order:   ", pending["order_id"])
    print("  amount:  ", pending["amount"])
    print("  reason:  ", pending["reason"])

    print("\nrefunds issued while paused:", ledger())
    print("told the customer:          ", flow.state.reply.strip()[:200])

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastApprovalFlow                                                                                     │
│  ID: 5918364b-7c9e-4877-a6c5-5a04f1bfbee9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

paused for approval: True
flow id:             5918364b-7c9e-4877-a6c5-5a04f1bfbee9
pending refund:
  order:    A1003
  amount:   250.0
  reason:   The order is delayed beyond the ETA and the customer is requesting a refund.

refunds issued while paused: no refund
told the customer:           A supervisor must approve this refund. You will hear back shortly. We apologize for the inconvenience caused by the delay.


## The supervisor approves

`resume` restores the flow **from its saved state**, by id, and continues it with the decision. That is
deliberate: the supervisor's answer could come hours later, from another process. Here the saved state
is a SQLite file inside the tutorial, `.crewai_flows/flows.db`. In production it must live in a database
every server can reach (Postgres or Redis, through your own `FlowPersistence`), so whichever server gets
the answer can resume the flow. Code then issues exactly the amount the supervisor saw; the model does
not get another chance to change it.

In [7]:
# Resume the saved flow by id with the supervisor's decision.
if paused is None:
    print(SKIPPED)
else:
    approved = await resume(paused.context.flow_id, "approve")

    print("refunds issued:", ledger())
    print("flow's reply:  ", approved.state.reply)

refunds issued: [('A1003', 250.0)]
flow's reply:   Good news: your refund of $250.00 for order A1003 has been approved and issued.


## The supervisor rejects

Same message, new flow, and this time `reject`. No refund is issued; the agent is told, and policy P5
says a rejected refund goes to a human agent.

In [8]:
shipfast.reset()
flow, paused = await run_until_refund_attempt()  # runs until the flow pauses for approval

print("paused for approval:", paused is not None)
if paused is not None:
    print("flow id:            ", paused.context.flow_id)

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastApprovalFlow                                                                                     │
│  ID: 01f470d7-7e5c-43fe-877e-0c2b222952aa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

paused for approval: True
flow id:             01f470d7-7e5c-43fe-877e-0c2b222952aa


The flow is paused and saved again. Resume it with `reject`, then check what happened: no refund, a
ticket for a human agent, and the tools the agent called.

In [9]:
if paused is None:
    print(SKIPPED)
else:
    rejected = await resume(paused.context.flow_id, "reject")

    print("refunds issued:", ledger())
    print("tickets opened:", ", ".join(t["ticket_id"] for t in shipfast.tickets()) or "none")
    print("tools called:  ", ", ".join(shipfast.tool_calls()) or "none")
    print("flow's reply:  ", rejected.state.reply.strip()[:200])

refunds issued: no refund
tickets opened: TCK-A1003
tools called:   lookup_order, track_shipment, search_policy, escalate_to_human
flow's reply:   A specialist has been assigned to your case, ticket TCK-A1003. We will follow up with you shortly. We apologize for the inconvenience caused by the delay.


The refund cannot happen without `approve`, because the gated tool never calls the real one above
$100, and after approval it is code, not the model, that issues the refund. The escalation after a
rejection *is* left to the model, and ShipFast's tests check it: scenario S3b expects `escalated`.

## How the others do this

- **LangGraph** pauses inside the tool call itself: middleware wraps every tool call and calls
  `interrupt()` before a refund over $100; `Command(resume="approve")` re-runs the same call. The agent
  stays one loop, and a checkpointer (in memory is enough within one process) holds the paused state.
- **Strands** can interrupt from a hook on `BeforeToolCallEvent`, which fires before each tool call;
  the hook checks the refund amount and pauses, closer to LangGraph's shape than to a Flow step.

## Takeaways

- A prompt asks; a guardrail enforces. Put money-moving checks in code.
- CrewAI pauses between Flow steps with `@human_feedback`; a non-blocking provider turns that into a pause
  the caller resumes later.
- CrewAI resumes only from saved state, so the flow needs `@persist`.
- Because the pause sits between steps, the refund itself becomes a step, and code issues the approved
  amount.
- Next: **07 · Tracing and eval**, measuring runs the same way in all three frameworks.